# Example 4.6 / 2: Power Sets

Companion notebook for **4.6 Collections of Sets and Power Sets** in the AMPL lab Documentation, section "Power Sets". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Power Sets

The power set of a set with $n$ members is the collection of all its $2^n$ subsets, from the
empty set to the whole set. AMPL has no power-set operator, but an indexed collection can build
one. Number the subsets $k = 0, \dots, 2^n - 1$ and let the binary digits of $k$ choose the
members: item $i$ belongs to subset $k$ when digit $\text{ord}(i) - 1$ of $k$ is 1. That
digit is `(k div 2^(ord(i) - 1)) mod 2`, using the operators `div`, `mod` and `^` from
Arithmetic Operators (AMPL lab documentation, section 5.1). `ITEMS` must be ordered, so that `ord(i)` is defined:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ITEMS ordered;
param n = card(ITEMS);
set SUBSETS = 0 .. 2^n - 1;
set POWER {k in SUBSETS} = {i in ITEMS: (k div 2^(ord(i) - 1)) mod 2 = 1};

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ITEMS"] = ["bands", "coils", "plate"]

In [5]:
# run.py
for k in ampl.set["SUBSETS"].to_list():
    print(int(k), ampl.set["POWER"][k].to_list())

0 []
1 ['bands']
2 ['coils']
3 ['bands', 'coils']
4 ['plate']
5 ['bands', 'plate']
6 ['coils', 'plate']
7 ['bands', 'coils', 'plate']


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set ITEMS := bands coils plate;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display POWER;

set POWER[0] := ; # empty
set POWER[1] := bands;
set POWER[2] := coils;
set POWER[3] := bands coils;
set POWER[4] := plate;
set POWER[5] := bands plate;
set POWER[6] := coils plate;
set POWER[7] := bands coils plate;

